In [13]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import json
import pandas as pd
from pathlib import Path

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from collections import Counter
import re
import numpy as np
import gensim.downloader as api
from sklearn.metrics import f1_score

# --- Preprocessing function ---
def preprocess_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"@\w+", '', text)
    text = re.sub(r"#\w+", '', text)
    return text

# --- Dataset Class ---
class SentimentDataset(Dataset):
    def __init__(self, df, vocab, max_length, label_columns):
        self.texts = df['text'].values
        self.labels = df[label_columns].astype(int).values
        self.vocab = vocab
        self.max_length = max_length
        self.pad_token_id = self.vocab.get('<PAD>', 0)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        tokens = self.tokenize_text(text)
        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }

    def tokenize_text(self, text):
        words = re.findall(r'\b\w+\b', text.lower())
        tokens = [self.vocab.get(word, self.vocab.get('<UNK>', 1)) for word in words]

        if len(tokens) < self.max_length:
            tokens.extend([self.pad_token_id] * (self.max_length - len(tokens)))
        else:
            tokens = tokens[:self.max_length]
        return tokens

# --- Vocabulary building with gensim glove-twitter ---
def build_vocab(texts, word2vec_model=None, min_freq=2, embedding_dim=200):
    all_words = []
    for text in texts:
        words = re.findall(r'\b\w+\b', str(text).lower())
        all_words.extend(words)

    word_counts = Counter(all_words)

    vocab = {'<PAD>': 0, '<UNK>': 1}
    embeddings = [np.zeros(embedding_dim, dtype=np.float32),  # PAD embedding
                  np.random.normal(scale=0.6, size=embedding_dim).astype(np.float32)]  # UNK embedding

    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
            if word2vec_model and word in word2vec_model.key_to_index:
                embeddings.append(word2vec_model[word].astype(np.float32))
            else:
                embeddings.append(np.random.normal(scale=0.6, size=embedding_dim).astype(np.float32))

    embedding_matrix = torch.tensor(np.array(embeddings), dtype=torch.float32)
    return vocab, embedding_matrix

# --- Training function ---
def train_model(model, train_df, eval_df, device,
                max_length=100, epochs=5, batch_size=32, lr=0.001,
                weight_decay=1e-5, clip_grad_norm=1.0):

    vocab = build_vocab(train_df['text'])[0]
    pad_token_id = vocab.get('<PAD>', 0)

    train_dataset = SentimentDataset(train_df, vocab, max_length)
    eval_dataset = SentimentDataset(eval_df, vocab, max_length)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    model.to(device)
    print(f"Training on {device}...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].unsqueeze(1).to(device)

            optimizer.zero_grad()
            logits = model(input_ids)
            loss = criterion(logits, labels)
            loss.backward()

            if clip_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)

            optimizer.step()
            train_loss += loss.item()

        model.eval()
        eval_loss = 0
        correct = 0
        total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in eval_loader:
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].unsqueeze(1).to(device)

                logits = model(input_ids)
                loss = criterion(logits, labels)
                eval_loss += loss.item()

                probs = torch.sigmoid(logits)
                predicted = (probs > 0.5).float()

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Compute F1 score
        f1 = f1_score(all_labels, all_preds, average='binary')  # Binary F1
        macro_f1 = f1_score(all_labels, all_preds, average='macro')  # Macro F1
        per_class_f1 = f1_score(all_labels, all_preds, average=None)  # Returns [F1_class_0, F1_class_1]

        print(f'Epoch {epoch+1}/{epochs}:')
        print(f'  Train Loss: {train_loss/len(train_loader):.4f}')
        print(f'  Eval Loss: {eval_loss/len(eval_loader):.4f}')
        print(f'  Eval Accuracy: {100*correct/total:.2f}%')
        print(f'  Eval F1 Score (binary): {f1:.4f}')
        print(f'  Eval F1 Score (macro):  {macro_f1:.4f}')
        print(f'  Per-Class F1 Scores: class 0 = {per_class_f1[0]:.4f}, class 1 = {per_class_f1[1]:.4f}')
        print("-" * 30)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes, 
                 filter_sizes=(3, 4, 5), num_filters=100, dropout_prob=0.5, 
                 pretrained_embeddings=None, freeze_embeddings=False):
        super().__init__()

        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=freeze_embeddings)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, 
                      out_channels=num_filters, 
                      kernel_size=fs)
            for fs in filter_sizes
        ])

        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(len(filter_sizes) * num_filters, num_classes - 1)

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = embedded.permute(0, 2, 1) 
        conved = [F.relu(conv(embedded)) for conv in self.convs]
        pooled = [F.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]
        cat = self.dropout(torch.cat(pooled, dim=1))
        output = self.fc(cat)
        return output

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
import gensim.downloader as api

def preprocess_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"@\w+", '', text)
    text = re.sub(r"#\w+", '', text)
    #text = re.sub(r"[^a-z\s]", '', text)
    #tokens = nltk.word_tokenize(text)
    return text


df_t = df_train.copy()
df_e = df_eval.copy()

df_t['text'] = df_t['text'].apply(preprocess_tweet)
df_e['text'] = df_e['text'].apply(preprocess_tweet)

# Build vocab and embedding matrix using pretrained vectors
vocab, embedding_matrix = build_vocab(df_t['text'], word2vec_model=word2vec, embedding_dim=100)

# Instantiate model with pretrained embeddings, freezing embeddings
model = TextCNN(vocab_size=len(vocab),
                embedding_dim=embedding_matrix.shape[1],
                num_classes=2,
                pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
                freeze_embeddings=False)

# Train model
train_model(model, df_t, df_e, device=torch.device('cpu'), epochs=10)

/var/folders/0s/37b847w521n_1v4k0x2wrrlr0000gn/T/ipykernel_39116/1510613842.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),


Training on cpu...
Epoch 1/10:
  Train Loss: 0.6011
  Eval Loss: 0.4965
  Eval Accuracy: 76.80%
  Eval F1 Score (binary): 0.6962
  Eval F1 Score (macro):  0.7543
  Per-Class F1 Scores: class 0 = 0.8124, class 1 = 0.6962
------------------------------
Epoch 2/10:
  Train Loss: 0.4286
  Eval Loss: 0.4677
  Eval Accuracy: 78.38%
  Eval F1 Score (binary): 0.7176
  Eval F1 Score (macro):  0.7712
  Per-Class F1 Scores: class 0 = 0.8248, class 1 = 0.7176
------------------------------
Epoch 3/10:
  Train Loss: 0.3561
  Eval Loss: 0.4287
  Eval Accuracy: 80.63%
  Eval F1 Score (binary): 0.7584
  Eval F1 Score (macro):  0.7984
  Per-Class F1 Scores: class 0 = 0.8383, class 1 = 0.7584
------------------------------
Epoch 4/10:
  Train Loss: 0.2860
  Eval Loss: 0.4293
  Eval Accuracy: 80.41%
  Eval F1 Score (binary): 0.7728
  Eval F1 Score (macro):  0.8003
  Per-Class F1 Scores: class 0 = 0.8277, class 1 = 0.7728
------------------------------
Epoch 5/10:
  Train Loss: 0.2152
  Eval Loss: 0.4451


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import sys
import pandas as pd

sys.path.append('/Users/lucfaessler/Documents/Studium/Bachelor/Wirtschaftsinformatik/Semester 8 - SS25/Practical Course NLP/nlp_practical_2025_sEXism/code')
sys.path.append('/Users/lucfaessler/Documents/Studium/Bachelor/Wirtschaftsinformatik/Semester 8 - SS25/Practical Course NLP/nlp_practical_2025_sEXism/data')
from data_loader.single_task_dataset import SingleTaskDataset
from data_loader.vocab import build_vocab
from train.torch_train import train_model
from preprocessing.utils import preprocess_dataframe
from models.new.pytorch.vanilla_models import textcnn, bilstm, bigru
from models.new.pytorch.attention_models import *
from embeddings.utils import *


df1 = pd.read_csv("../../../../data/en/encoded/aeda_encoded.csv")
df2 = pd.read_csv("../../../../data/en/encoded/backtranslated_encoded.csv")
df3 = pd.read_csv("../../../../data/en/encoded/translated_encoded.csv")

df_train = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

df_train = preprocess_dataframe(df_train, 'tweet')

label_column = 'gold_labels_task1_1'

df_train = df_train[df_train[label_column].notna()]

# 1. Preprocess text column

# 2. Split train into train+val
train_df, val_df = train_test_split(df_train, test_size=0.1, stratify=df_train[label_column])

embedding = load_embeddings("glove-twitter-200")

# 3. Build vocab and embedding matrix
vocab, embedding_matrix = build_vocab(train_df["tweet"], embedding=embedding, embedding_dim=200)

# 4. Create Dataset and DataLoader
train_ds = SingleTaskDataset(train_df, vocab, max_length=100, label_column=label_column, task='binary')
val_ds   = SingleTaskDataset(val_df,   vocab, max_length=100, label_column=label_column, task='binary')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)

print(train_ds.__getitem__(0))

{'input_ids': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0]), 'labels': tensor(0.)}


In [20]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import sys
import pandas as pd
import numpy as np

sys.path.append('/Users/lucfaessler/Documents/Studium/Bachelor/Wirtschaftsinformatik/Semester 8 - SS25/Practical Course NLP/nlp_practical_2025_sEXism/code')
sys.path.append('/Users/lucfaessler/Documents/Studium/Bachelor/Wirtschaftsinformatik/Semester 8 - SS25/Practical Course NLP/nlp_practical_2025_sEXism/data')
from data_loader.single_task_dataset import SingleTaskDataset
from data_loader.vocab import build_vocab
from train.torch_train import train_model
from preprocessing.utils import preprocess_dataframe
from models.new.pytorch.vanilla_models import textcnn, bilstm, bigru
from models.new.pytorch.attention_models import bigru, bilstm
from embeddings.utils import *


df1 = pd.read_csv("../../../../data/en/encoded/aeda_encoded.csv")
df2 = pd.read_csv("../../../../data/en/encoded/backtranslated_encoded.csv")
df3 = pd.read_csv("../../../../data/en/encoded/translated_encoded.csv")
df4 = pd.read_csv("../../../../data/en/encoded/training_encoded.csv")
df5 = pd.read_csv("../../../../data/en/encoded/dev_encoded.csv")

#print(df5[:10])

df_train = pd.concat([df1, df2, df3, df4], axis=0, ignore_index=True)

df_train = preprocess_dataframe(df_train, 'tweet')

df_val = preprocess_dataframe(df5, 'tweet')

label_column = 'gold_labels_task1_1'

df_train = df_train[df_train[label_column].notna()]
val_df = df_val[df_val[label_column].notna()]

#print(val_df[:10])

# 1. Preprocess text column

# 2. Split train into train+val
#train_df, val_df = train_test_split(df_train, test_size=0.1, stratify=df_train[label_column])

#embedding = load_embeddings("glove-twitter-200")
embedding = load_embeddings("glove-twitter-100")

# 3. Build vocab and embedding matrix
vocab, embedding_matrix = build_vocab(train_df["tweet"], embedding=embedding, embedding_dim=100)

# 4. Create Dataset and DataLoader
train_ds = SingleTaskDataset(train_df, vocab, max_length=100, label_column=label_column, task='binary')
val_ds   = SingleTaskDataset(val_df,   vocab, max_length=100, label_column=label_column, task='binary')
#train_ds = MultiTaskDataset(train_df, vocab, max_length=100, label_column=label_column)
#val_ds   = MultiTaskDataset(val_df,   vocab, max_length=100, label_column=label_column)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)

#model = textcnn.TextCNN(
#    vocab_size=len(vocab),
#    embedding_dim=embedding_matrix.shape[1],
#    num_classes=1,
#    pretrained_embeddings=embedding_matrix
#)

#model = textcnn.TextCNN(vocab_size=len(vocab),
#                embedding_dim=embedding_matrix.shape[1],
#                num_classes=4,
#                pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
#                freeze_embeddings=False)

model = bilstm.BiLSTMAttentionClassifier(vocab_size=len(vocab), embedding_dim=embedding_matrix.shape[1], hidden_dim=128, num_classes=2, 
                 dropout_prob=0.5, pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float), freeze_embeddings=False)

# 6. Loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.9)

# 7. Train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    loss_fn,
    device,
    task_type='binary',
    epochs=15,
    use_wandb=False,  # change to True to enable Weights & Biases
    run_name="textcnn_multiclass"
)


/var/folders/0s/37b847w521n_1v4k0x2wrrlr0000gn/T/ipykernel_97005/2289960202.py:75: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dropout_prob=0.5, pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float), freeze_embeddings=False)
/opt/miniconda3/envs/gensim-env/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


[Epoch 1/15]
Train Loss: 0.4679
  Train Accuracy: 77.11%
  Train F1 Score (macro): 0.7280
  Per-Class F1 Scores: class 0 = 0.8362, class 1 = 0.6197

Val Loss: 0.4233
  Val Accuracy: 80.63%
  Val F1 Score (macro): 0.8022
  Per-Class F1 Scores: class 0 = 0.8307, class 1 = 0.7737
--------------------------------------------------
[Epoch 2/15]
Train Loss: 0.2951
  Train Accuracy: 87.73%
  Train F1 Score (macro): 0.8646
  Per-Class F1 Scores: class 0 = 0.9060, class 1 = 0.8233

Val Loss: 0.5275
  Val Accuracy: 74.77%
  Val F1 Score (macro): 0.7264
  Per-Class F1 Scores: class 0 = 0.8028, class 1 = 0.6500
--------------------------------------------------
[Epoch 3/15]
Train Loss: 0.1736
  Train Accuracy: 93.51%
  Train F1 Score (macro): 0.9294
  Per-Class F1 Scores: class 0 = 0.9495, class 1 = 0.9093

Val Loss: 0.6555
  Val Accuracy: 76.80%
  Val F1 Score (macro): 0.7578
  Per-Class F1 Scores: class 0 = 0.8075, class 1 = 0.7082
--------------------------------------------------
[Epoch 4/15]


In [3]:
print("Train label distribution:", train_df[label_column].value_counts(normalize=True))
print("Val label distribution:", val_df[label_column].value_counts(normalize=True))

Train label distribution: gold_labels_task1_2
0.0    0.636411
1.0    0.222107
2.0    0.078866
3.0    0.062616
Name: proportion, dtype: float64
Val label distribution: gold_labels_task1_2
0.0    0.636872
1.0    0.221601
2.0    0.079143
3.0    0.062384
Name: proportion, dtype: float64


In [3]:
label = [1, 0, 0, 0, 0, 0]
test = torch.tensor(label, dtype=torch.float)
test

tensor([1., 0., 0., 0., 0., 0.])

tensor([0.7056, 0.4588, 0.5445, 0.7401, 0.7068, 0.7438, 0.6811, 0.6152, 0.3915,
        0.5779, 0.5118, 0.6343, 0.4743, 0.6231, 0.6613, 0.6669, 0.6871, 0.6783,
        0.7819, 0.5746, 0.5341, 0.6451, 0.5528, 0.5710, 0.5064])
tensor([1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
        0])


In [11]:
import pandas as pd
df5 = pd.read_csv("../../../../data/en/encoded/dev_encoded.csv")
df_val = preprocess_dataframe(df5, 'tweet')
df_val[:10]


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,id_EXIST,lang,tweet,number_annotators,annotators,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,labels_task1_1,labels_task1_2,labels_task1_3,split,gold_labels_task1_1,gold_labels_task1_2,gold_labels_task1_3
0,0,549,549,400001,en,"“you should smile more, love. just pretend you...",6,"['Annotator_764', 'Annotator_765', 'Annotator_...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '18-22', '23-45', '4...","['Hispano or Latino', 'Asian', 'White or Cauca...","['High school degree or equivalent', 'Bachelor...","['Mexico', 'Canada', 'Hungary', 'Poland', 'Spa...","['NO', 'NO', 'NO', 'NO', 'YES', 'YES']","['-', '-', '-', '-', 'REPORTED', 'DIRECT']","[['-'], ['-'], ['-'], ['-'], ['IDEOLOGICAL-INE...",DEV_EN,0.0,0.0,"[1, 0, 0, 0, 0, 0]"
1,1,550,550,400002,en,she is right but the push is all in the opposi...,6,"['Annotator_770', 'Annotator_771', 'Annotator_...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '18-22', '23-45', '4...","['Black or African American', 'Black or Africa...","['Bachelor’s degree', 'Master’s degree', 'Bach...","['South Africa', 'South Africa', 'Canada', 'Es...","['YES', 'YES', 'NO', 'YES', 'YES', 'NO']","['REPORTED', 'JUDGEMENTAL', '-', 'REPORTED', '...","[['IDEOLOGICAL-INEQUALITY'], ['OBJECTIFICATION...",DEV_EN,1.0,2.0,"[0, 1, 0, 0, 1, 0]"
2,2,551,551,400003,en,some man moving my suitcase in the overhead lu...,6,"['Annotator_776', 'Annotator_777', 'Annotator_...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '18-22', '23-45', '4...","['Multiracial', 'White or Caucasian', 'White o...","['High school degree or equivalent', 'High sch...","['South Africa', 'Finland', 'Portugal', 'Polan...","['NO', 'YES', 'YES', 'YES', 'YES', 'YES']","['-', 'REPORTED', 'REPORTED', 'REPORTED', 'REP...","[['-'], ['STEREOTYPING-DOMINANCE'], ['OBJECTIF...",DEV_EN,1.0,2.0,"[0, 0, 0, 1, 0, 0]"
3,3,552,552,400004,en,"lol gamergate the go to boogieman, maybe if th...",6,"['Annotator_780', 'Annotator_781', 'Annotator_...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '18-22', '23-45', '4...","['Hispano or Latino', 'Hispano or Latino', 'Wh...","['High school degree or equivalent', 'Bachelor...","['Chile', 'Mexico', 'New Zealand', 'Mexico', '...","['YES', 'NO', 'NO', 'NO', 'NO', 'NO']","['DIRECT', '-', '-', '-', '-', '-']","[['STEREOTYPING-DOMINANCE', 'OBJECTIFICATION',...",DEV_EN,0.0,0.0,"[1, 0, 0, 0, 0, 0]"
4,4,553,553,400005,en,to me this has the same negativity as gamergat...,6,"['Annotator_780', 'Annotator_781', 'Annotator_...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '18-22', '23-45', '4...","['Hispano or Latino', 'Hispano or Latino', 'Wh...","['High school degree or equivalent', 'Bachelor...","['Chile', 'Mexico', 'New Zealand', 'Mexico', '...","['YES', 'NO', 'NO', 'YES', 'NO', 'NO']","['JUDGEMENTAL', '-', '-', 'REPORTED', '-', '-']","[['IDEOLOGICAL-INEQUALITY'], ['-'], ['-'], ['M...",DEV_EN,0.0,0.0,"[1, 0, 0, 0, 0, 0]"
5,5,554,554,400006,en,"yeah it was a core ""meme"" in the pre gamergate...",6,"['Annotator_780', 'Annotator_781', 'Annotator_...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '18-22', '23-45', '4...","['Hispano or Latino', 'Hispano or Latino', 'Wh...","['High school degree or equivalent', 'Bachelor...","['Chile', 'Mexico', 'New Zealand', 'Mexico', '...","['YES', 'NO', 'NO', 'NO', 'NO', 'NO']","['DIRECT', '-', '-', '-', '-', '-']","[['STEREOTYPING-DOMINANCE', 'MISOGYNY-NON-SEXU...",DEV_EN,0.0,0.0,"[1, 0, 0, 0, 0, 0]"
6,6,555,555,400007,en,showing off? the man spending his money helps ...,6,"['Annotator_785', 'Annotator_786', 'Annotator_...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '18-22', '23-45', '4...","['White or Caucasian', 'Hispano or Latino', 'W...","['Bachelor’s degree', 'Bachelor’s degree', 'Hi...","['Poland', 'Mexico', 'United Kingdom', 'Sweden...","['NO', 'NO', 'NO', 'NO', 'NO', 'NO']","['-', '-', '-', '-', '-', '-']",